# S3 Parquet Loader — Getting Started

This notebook walks through **every** use-case of the `s3dataloader` library, from
basic setup to advanced engine configuration. Replace collection names, column
names, and dates with values that exist in your S3 dumps.

**Table of contents**

1. [Setup & Environment](#1-setup--environment)
2. [S3ParquetLoader — Core API](#2-s3parquetloader--core-api)
   - 2.1 Simple column load (no date filter)
   - 2.2 Column load with date filter
   - 2.3 Custom SQL (aggregations, filters, joins)
   - 2.4 Custom SQL on indexed collections
   - 2.5 Per-call source / prefix override
3. [S3IndexedCollectionReader — Direct Usage](#3-s3indexedcollectionreader--direct-usage)
   - 3.1 Read with date-range chunk pruning
   - 3.2 Read without date filter (full scan)
   - 3.3 Custom SQL with chunk pruning
   - 3.4 Inspect schema columns
4. [Indexed Collections Registry](#4-indexed-collections-registry)
5. [DataLoader — Engine-Agnostic Facade](#5-dataloader--engine-agnostic-facade)
   - 5.1 DuckDB engine (default)
   - 5.2 Athena engine
   - 5.3 Wrangler engine
   - 5.4 Engine comparison at a glance
6. [DataLoader API Methods](#6-dataloader-api-methods)
   - 6.1 `read_table` — load a collection
   - 6.2 `read_sql` — execute SQL
   - 6.3 `read_parquet` — direct S3 parquet reads
   - 6.4 `scan` — chunked / streaming iteration
   - 6.5 `load_dataframe` — alias for `read_table`
7. [Error Handling](#7-error-handling)
8. [Advanced Patterns](#8-advanced-patterns)
   - 8.1 Registering a custom engine
   - 8.2 Enabling debug logging
   - 8.3 S3 prefix resolution helpers
   - 8.4 Date normalization utilities
9. [Tips & Gotchas](#9-tips--gotchas)

---
## 1. Setup & Environment

### Install dependencies

```bash
# From the project root
python3 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt          # or: pip install -e .

# For the Athena and Wrangler engines (optional)
pip install "s3dataloader[aws]"           # installs awswrangler>=3.4
```

### Configure `.env`

Copy `.env.example` → `.env` and fill in your S3 settings. **Never commit `.env`.**
Use your own AWS access keys in `.env` — that is the only credential source for local use.

```env
S3_BUCKET=airflow-etl-db

# Named dump prefixes (preferred)
S3_PREFIX_LEADS_DB=production_raw_dump/leads-db
S3_PREFIX_CRM=production_raw_dump/crm

AWS_REGION=ap-south-1

# Your AWS credentials (required for local use)
AWS_ACCESS_KEY_ID=...
AWS_SECRET_ACCESS_KEY=...
# AWS_SESSION_TOKEN=...   # only for temporary ASIA… keys; omit for AKIA…

# Athena engine defaults (optional)
# ATHENA_DATABASE=leads-prod
# ATHENA_WORKGROUP=asbl_analytics_prod
# ATHENA_OUTPUT_LOCATION=s3://airflow-etl-db/athena_storage/
```

> **Important:** Run Python from the **project root** so `dataSource` imports resolve.
> Restart the kernel after changing `.env`.


In [1]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)  # pick up AWS_* / S3_* from .env

from dataSource import (
    S3ParquetLoader,
    S3IndexedCollectionReader,
    INDEXED_COLLECTIONS,
    DataLoader,
    LoaderFactory,
    resolve_s3_prefix,
)
from dataSource.aws_auth import resolve_aws_auth, create_s3_client
import dataSource.getData as _getData

# Fresh DuckDB connection picks up current credentials
_getData._CONNECTION = None

auth = resolve_aws_auth()
assert auth.has_credentials, (
    "No AWS credentials found. Set AWS_ACCESS_KEY_ID and "
    "AWS_SECRET_ACCESS_KEY in .env (see .env.example)."
)

bucket = os.getenv("S3_BUCKET", "").strip()
assert bucket, "S3_BUCKET must be set in .env"

try:
    create_s3_client().list_objects_v2(
        Bucket=bucket,
        Prefix="production_raw_dump/leads-db/",
        MaxKeys=1,
    )
except Exception as exc:
    raise RuntimeError(
        "S3 auth failed with the credentials in .env "
        f"({type(exc).__name__}: {exc}).\n"
        "Fix AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY "
        "(omit AWS_SESSION_TOKEN for AKIA… permanent keys), "
        "then restart the kernel and re-run this cell."
    ) from exc

print(f"✅ All imports successful (region={auth.region}, bucket={bucket})")


✅ All imports successful (region=ap-south-1, bucket=airflow-etl-db)


---
## 2. S3ParquetLoader — Core API

`S3ParquetLoader` is the **original, battle-tested** entry point. It reads Airflow
ETL parquet dumps from S3 via DuckDB and returns pandas DataFrames.

### Named sources

| `source=`       | Env key              | Typical prefix                   |
|-----------------|----------------------|----------------------------------|
| `"leads-db"`    | `S3_PREFIX_LEADS_DB` | `production_raw_dump/leads-db`   |
| `"crm"`         | `S3_PREFIX_CRM`      | `production_raw_dump/crm`        |
| *(any other)*   | pass `s3_prefix=`    | *(your custom S3 prefix)*        |

In [2]:
# Create loaders for different data sources
leads_loader = S3ParquetLoader(source="leads-db")      # default
crm_loader   = S3ParquetLoader(source="crm")
custom_loader = S3ParquetLoader(s3_prefix="production_raw_dump/some-other-dump")

# S3ParquetLoader() with no args defaults to source="leads-db"
default_loader = S3ParquetLoader()

print("✅ Loaders created")

✅ Loaders created


### 2.1 Simple column load (no date filter)

Use `loader()` with a collection name and column list. This returns **all rows**
from the collection, projected to the requested columns.

In [3]:
loader = S3ParquetLoader(source="leads-db")

# Full collection load — no date filter
df = loader.loader(
    collection_name="projects",
    column_list=["_id", "createdAt"],
)

print(f"Shape: {df.shape}")
df.head()

Shape: (12, 2)


,_id,createdAt
0,5f9ff649ffef1e546c79ce9d,NaT
1,60dd82c75aec32505d9af7d5,NaT
2,667d2f5c2eb25893ac4e5c10,NaT
3,5f901dc0e854b05a8b790e25,NaT
4,667d2faddf713e02b9d3e822,NaT


### 2.2 Column load with date filter

Pass `date_column`, `start_date`, and `end_date` **all together** (all-or-none).
The date range is **inclusive**. Prefer ISO `YYYY-MM-DD` format.

In [7]:
df = loader.loader(
    collection_name="leads",
    column_list=["_id", "createdAt"],
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Rows in date range: {len(df)}")
df.head()

Rows in date range: 3098


,_id,createdAt
0,683bdc6a6e1744052c3dc2a7,2025-06-01 04:51:54.162
1,683be632cefdec1519cf80ff,2025-06-01 05:33:38.574
2,683beafc01c8232ff723360b,2025-06-01 05:54:04.065
3,683bebb627f712a6ed29970f,2025-06-01 05:57:10.733
4,683bed65c22d1258fc89faed,2025-06-01 06:04:21.243


> **Key rules:**
> - `date_column`, `start_date`, `end_date` — pass all three or none.
>   Passing one or two raises `ValueError`.
> - Accepts strings, `datetime`, or `pandas.Timestamp` (inclusive).
> - Missing columns are **skipped** with a console warning; check output.
> - Works for both flat *and* indexed collections — indexed ones use
>   manifest chunk pruning automatically.

### 2.3 Custom SQL (aggregations, filters, joins)

Use `custom_loader()` with `{parquet_path}` as a placeholder in your SQL.
The loader substitutes the correct S3 glob or pruned chunk list.

In [8]:
df = loader.custom_loader(
    sql_query="""
        SELECT masterLeadId, COUNT(*) AS n
        FROM parquet_scan('{parquet_path}')
        WHERE project = '5f9ff649ffef1e546c79ce9d'
        GROUP BY masterLeadId
    """,
    collection_name="leads",
)

print(f"Active leads: {len(df)}")
df.head()

Active leads: 59138


,masterLeadId,n
0,22488,1
1,28559,1
2,15672,1
3,28672,1
4,15605,1


### 2.4 Custom SQL on indexed collections

For **indexed** collections (e.g. `whatsappchatmessages`, `visitorlogs`),
pass dates so only overlapping chunks are read — much faster than full scans.

In [11]:
df = loader.custom_loader(
    sql_query="""
        SELECT lead, message, time
        FROM parquet_scan('{parquet_path}')
        WHERE message IS NOT NULL
    """,
    collection_name="whatsappchatmessages",
    date_column="time",
    start_date="2026-06-01",
    end_date="2026-06-03",
)

print(f"Messages in range: {len(df)}")
df.head()

: 

: 

> **Note:** Without dates on indexed collections, *all* chunks are globbed.
> This is fine for narrow `WHERE leadId = '...'` queries, but slow for full scans.

### 2.5 Per-call source / prefix override

Both `loader()` and `custom_loader()` accept `source=` / `s3_prefix=` to
override the constructor defaults for a single call.

In [5]:
# One loader, multiple sources per-call
loader = S3ParquetLoader(source="leads-db")

# Override source for one call — reads from CRM
df_crm = loader.loader(
    collection_name="projects",
    column_list=["_id", "name"],
    source="crm",
)

# Override with an arbitrary S3 prefix
df_other = loader.loader(
    collection_name="flats",
    column_list=["_id","createdAt"],
    s3_prefix="production_raw_dump/homes-prod",
)

print(f"CRM contacts: {len(df_crm)}, Other events: {len(df_other)}")

CRM contacts: 7, Other events: 2469


---
## 3. S3IndexedCollectionReader — Direct Usage

For large, chunked collections stored under `{prefix}/{collection}/data/*.parquet`
with a JSON manifest at `{prefix}/{collection}/_index/{collection}_index.json`,
you can use the reader directly.

The `from_env()` factory reads `S3_BUCKET` from the environment automatically.

### 3.1 Read with date-range chunk pruning

In [7]:
from dataSource import S3IndexedCollectionReader

reader = S3IndexedCollectionReader.from_env("visitorlogs", database="leads-db")

# With date filter — uses chunk pruning + row filter
df = reader.read_range(
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
    columns=["globalId", "createdAt"],
)

print(f"Visitor logs in range: {len(df)}")
df.head()

Visitor logs in range: 134702


,globalId,createdAt
0,None,2025-06-01 08:28:17.174
1,None,2025-06-01 08:28:33.942
2,None,2025-06-01 08:28:33.716
3,None,2025-06-01 08:28:37.368
4,None,2025-06-01 08:28:46.384


### 3.2 Read without date filter (full scan)

In [ ]:
# Without dates — globs all chunks (no WHERE clause)
df = reader.read_range(
    columns=["globalId", "createdAt"],
)

print(f"Total visitor logs: {len(df)}")
df.head()

### 3.3 Custom SQL with chunk pruning

Use `read_sql()` with `{parquet_path}` for aggregations on indexed collections.
The reader substitutes the pruned chunk list.

In [8]:
df = reader.read_sql(
    """
    SELECT
    visitorId,
    MAX(createdAt) AS createdAt,
    COUNT(*) AS visits
    FROM parquet_scan('{parquet_path}')
    GROUP BY visitorId
    """,
    date_column="createdAt",
    start_date="2026-06-01",
    end_date="2026-06-30",
)

print(f"Leads with visits in June: {len(df)}")
df.head()

Leads with visits in June: 248605


,visitorId,createdAt,visits
0,Visitor-27520650,2026-06-23 04:42:39.093,1
1,Visitor-27497258,2026-06-23 04:51:07.699,3
2,Visitor-27521623,2026-06-25 01:55:38.488,3
3,Visitor-27522975,2026-06-23 18:05:11.009,2
4,Visitor-24737288,2026-06-24 04:02:09.559,1


> **Note:** No overlapping chunks → returns an **empty** DataFrame (not an error).

### 3.4 Inspect schema columns

Use `describe_columns()` to quickly check the parquet schema from the first chunk.

In [9]:
columns = reader.describe_columns()
print("Available columns:")
for col in columns:
    print(f"  • {col}")

Available columns:
  • _id
  • __v
  • active
  • botType
  • browser
  • callNowClicked
  • channel
  • chatDetails
  • country
  • createdAt
  • createdAtTime
  • cummulativeTimeSpent
  • endTimes
  • enteredDate
  • fbclid
  • fromRecoveryScript
  • gclid
  • geoLocFetchViaCoords
  • geoLocality
  • globalId
  • globalVisitorId
  • googleReviewsTimeSpent
  • instanceNumber
  • ipAddress
  • ipBasedGeoData
  • ipBasedLocationCoords
  • ipstackFromIpAddress
  • ipstackFromLocCoords
  • isActualMicroMarket
  • isRetargeting
  • isTimeSpentUpdated
  • lead
  • loanEligibility
  • locFetchViaIp
  • locationCoords
  • microMarket
  • multiplicationScore
  • onlineStatus
  • operatingSystem
  • pages
  • parentSourceCampaign
  • parent_utm_campaign
  • polls
  • pollsTimeSpent
  • previouslyAddedEmail
  • priceSheetsOpened
  • projectId
  • referrerUrl
  • requestedCallDateAndTime
  • secondaryLeads
  • secondaryUserDetails
  • sessionId
  • sourceCampaign
  • sourceUrl
  • sourceUser
  • 

---
## 4. Indexed Collections Registry

The library tracks which collections use the chunked/indexed layout.
When `S3ParquetLoader` detects one, it automatically delegates to
`S3IndexedCollectionReader` for manifest-based chunk pruning.

In [10]:
from dataSource import INDEXED_COLLECTIONS

print("Indexed (chunked) collections:")
for name in sorted(INDEXED_COLLECTIONS):
    print(f"  • {name}")

Indexed (chunked) collections:
  • flowbuilderleadtransactions
  • globalvisitorlogs
  • userredirecttracks
  • visitorlogs
  • whatsappchatmessages


> **Maintaining the list:** When data-engineering ships a new chunked dump,
> add its name to `INDEXED_COLLECTIONS` in
> `dataSource/hugeCollection_chunk_reading.py`.

---
## 5. DataLoader — Engine-Agnostic Facade

`DataLoader` is the **modern entry point** that unifies three backends behind
one API. Configure the engine once; the methods (`read_parquet`, `read_sql`,
`read_table`, `scan`, `load_dataframe`) stay identical.

### Why multiple engines?

S3 GET requests are rate-limited (~1,000/min) far below platform load.
Large reads should go through **Athena** (server-side SQL) instead of
direct S3 scans.

### 5.1 DuckDB engine (default)

Same behavior as `S3ParquetLoader`, including indexed-collection chunk pruning.
Best for ad-hoc reads that stay under the S3 GET request budget.

In [11]:
from dataSource import DataLoader

# DuckDB is the default engine
duckdb_loader = DataLoader()  # equivalent to DataLoader(engine="duckdb")

print(f"Engine: {duckdb_loader.engine_name}")

df = duckdb_loader.read_table(
    "leads",
    columns=["_id", "createdAt"],
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Leads loaded (DuckDB): {len(df)}")
df.head()

Engine: duckdb
Leads loaded (DuckDB): 3098


,_id,createdAt
0,683bdc6a6e1744052c3dc2a7,2025-06-01 04:51:54.162
1,683be632cefdec1519cf80ff,2025-06-01 05:33:38.574
2,683beafc01c8232ff723360b,2025-06-01 05:54:04.065
3,683bebb627f712a6ed29970f,2025-06-01 05:57:10.733
4,683bed65c22d1258fc89faed,2025-06-01 06:04:21.243


### 5.2 Athena engine

Runs SQL through AWS Athena via `awswrangler`. Preferred for large scans
since it avoids per-file S3 GET requests from the client.

Requires `awswrangler` (`pip install "s3dataloader[aws]"`).

| Setting              | Env fallback               |
|----------------------|----------------------------|
| `database=`          | `ATHENA_DATABASE`          |
| `workgroup=`         | `ATHENA_WORKGROUP`         |
| `output_location=`   | `ATHENA_OUTPUT_LOCATION`   |

In [12]:
athena_loader = DataLoader(
    engine="athena",
    database="leads-prod",
    workgroup="asbl_analytics_prod",
    output_location="s3://airflow-etl-db/athena_storage/",
)

print(f"Engine: {athena_loader.engine_name}")

# SQL via Athena
df = athena_loader.read_sql("SELECT * FROM projects")
print(f"Projects (Athena): {len(df)}")
df.head()

Engine: athena
Projects (Athena): 12


,name,id,is_actual_project
0,LAKESIDE,5f901dc0e854b05a8b790e25,True
1,BLOG,67d28a2a6a8ce0a509bd44bb,False
2,LOFT,64a5178ed9039b2988d6e4ab,True
3,BROADWAY,671353e397ece153c124590c,True
4,ASBL,667d2f5c2eb25893ac4e5c10,False


### 5.3 Wrangler engine

Bulk S3 parquet reads via `awswrangler.s3.read_parquet`. Supports
dataset mode, partition filters, chunking, and path lists.

Requires `awswrangler` (`pip install "s3dataloader[aws]"`).

In [ ]:
wrangler_loader = DataLoader(engine="wrangler", source="leads-db")

print(f"Engine: {wrangler_loader.engine_name}")

df = wrangler_loader.read_table(
    "visitorlogs",
    columns=["globalId", "createdAt"],
)

print(f"Leads loaded (Wrangler): {len(df)}")
df.head()

### 5.4 Engine comparison at a glance

| Engine    | Backed by                           | Best for                                            | Not supported    |
|-----------|-------------------------------------|-----------------------------------------------------|------------------|
| `duckdb`  | DuckDB `parquet_scan`               | Ad-hoc reads, indexed-collection chunk pruning      | —                |
| `athena`  | `awswrangler.athena.read_sql_query` | Large scans, SQL over Glue catalog (avoids S3 GETs) | `read_parquet`   |
| `wrangler`| `awswrangler.s3.read_parquet`       | Bulk dataset reads, partition filters, chunking      | `read_sql`       |

---
## 6. DataLoader API Methods

All five methods work with every engine (unsupported operations raise a
descriptive error pointing you to the right engine).

### 6.1 `read_table` — load a collection

The most common operation. Works across all three engines.

In [16]:
loader = DataLoader()  # DuckDB default

# Load specific columns
df = loader.read_table("leads", columns=["_id", "leadId"])
print(f"All leads: {len(df)}")
df.head()

All leads: 1267279


,_id,leadId
0,5fcb8732ad3f951c77e56654,115
1,5fd079aaad3f951c77e5a0e6,124
2,5fde08382987d55ff925e018,212
3,5fde44232987d55ff925e0b8,215
4,5fe339b30ffd1c4261605af2,246


In [17]:
# With date filter
df = loader.read_table(
    "leads",
    columns=["_id", "createdAt", "leadId"],
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Leads in date range: {len(df)}")
df.head()

Leads in date range: 3098


,_id,createdAt,leadId
0,683bdc6a6e1744052c3dc2a7,2025-06-01 04:51:54.162,703101
1,683be632cefdec1519cf80ff,2025-06-01 05:33:38.574,703112
2,683beafc01c8232ff723360b,2025-06-01 05:54:04.065,703118
3,683bebb627f712a6ed29970f,2025-06-01 05:57:10.733,703119
4,683bed65c22d1258fc89faed,2025-06-01 06:04:21.243,703124


In [ ]:
# Load all columns (no projection)
df = loader.read_table("leads")
print(f"All columns: {list(df.columns)}")
df.head()

### 6.2 `read_sql` — execute SQL

Supported by **DuckDB** and **Athena**. The wrangler engine raises `WranglerError`.

In [ ]:
# DuckDB: SQL with {parquet_path} placeholder
loader = DataLoader()  # DuckDB

df = loader.read_sql(
    "SELECT leadId, COUNT(*) AS n FROM parquet_scan('{parquet_path}') GROUP BY leadId",
    collection="whatsappchatmessages",
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Leads with messages: {len(df)}")
df.head()

In [18]:
# Athena: standard SQL against Glue catalog tables
athena_loader = DataLoader(
    engine="athena",
    database="leads-prod",
    workgroup="asbl_analytics_prod",
    output_location="s3://airflow-etl-db/athena_storage/",
)

df = athena_loader.read_sql("SELECT * FROM projects")
df.head()

,name,id,is_actual_project
0,LAKESIDE,5f901dc0e854b05a8b790e25,True
1,BLOG,67d28a2a6a8ce0a509bd44bb,False
2,LOFT,64a5178ed9039b2988d6e4ab,True
3,BROADWAY,671353e397ece153c124590c,True
4,ASBL,667d2f5c2eb25893ac4e5c10,False


### 6.3 `read_parquet` — direct S3 parquet reads

Supported by **DuckDB** and **Wrangler**. Athena raises `AthenaError`.

In [ ]:
# DuckDB: read parquet by path
loader = DataLoader()

df = loader.read_parquet(
    "s3://airflow-etl-db/production_raw_dump/leads-db/leads/*.parquet",
    columns=["_id"],
)

print(f"Lead IDs from parquet: {len(df)}")
df.head()

In [ ]:
# Wrangler: read parquet by path
wrangler_loader = DataLoader(engine="wrangler")

df = wrangler_loader.read_parquet(
    path="s3://airflow-etl-db/production_raw_dump/leads-db/leads/",
    columns=["_id", "createdAt"],
)

print(f"Lead IDs (Wrangler): {len(df)}")
df.head()

In [ ]:
# Wrangler: read parquet by collection name
df = wrangler_loader.read_parquet(
    collection="leads",
    columns=["_id", "createdAt"],
)

print(f"Leads by collection name: {len(df)}")
df.head()

### 6.4 `scan` — chunked / streaming iteration

Lazily stream a result in DataFrame chunks. Useful for processing large
datasets without loading everything into memory.

In [ ]:
# DuckDB scan: chunked iteration over SQL results
loader = DataLoader()

total_rows = 0
for chunk in loader.scan(
    "SELECT * FROM parquet_scan('{parquet_path}')",
    collection="leads",
):
    total_rows += len(chunk)
    print(f"  Chunk: {len(chunk)} rows")

print(f"Total rows processed: {total_rows}")

In [ ]:
# Athena scan: chunked SQL results
athena_loader = DataLoader(
    engine="athena",
    database="leads-prod",
    workgroup="asbl_analytics_prod",
    output_location="s3://airflow-etl-db/athena_storage/",
)

for chunk in athena_loader.scan(
    "SELECT * FROM big_table",
    chunksize=10_000,
):
    print(f"  Chunk: {len(chunk)} rows")
    # process(chunk)

In [ ]:
# Wrangler scan: chunked S3 parquet reads
wrangler_loader = DataLoader(engine="wrangler")

for chunk in wrangler_loader.scan(
    "visitorlogs",
    chunked=100_000,
):
    print(f"  Chunk: {len(chunk)} rows")
    # process(chunk)

### 6.5 `load_dataframe` — alias for `read_table`

Convenience alias; identical behavior.

In [ ]:
loader = DataLoader()

df = loader.load_dataframe(
    "leads",
    columns=["_id", "createdAt"],
    date_column="createdAt",
    start_date="2025-06-01",
    end_date="2025-06-07",
)

print(f"Leads via load_dataframe: {len(df)}")
df.head()

---
## 7. Error Handling

All engine failures are surfaced as subclasses of `LoaderError`, so you
can catch one base class regardless of the configured backend.

| Exception            | When raised                                      |
|----------------------|--------------------------------------------------|
| `LoaderError`        | Base class for all loader errors                 |
| `ConfigurationError` | Unknown engine, missing settings, invalid config |
| `DuckDBError`        | DuckDB query failure                             |
| `AthenaError`        | Athena query failure or unsupported operation    |
| `WranglerError`      | Wrangler S3 read failure or unsupported operation|

In [ ]:
from dataSource import LoaderError, ConfigurationError, AthenaError, WranglerError

# Example 1: Catch any loader error with the base class
try:
    loader = DataLoader()
    df = loader.read_sql("SELECT * FROM nonexistent_parquet_scan('bad/path')")
except LoaderError as exc:
    print(f"Caught LoaderError: {exc}")

In [ ]:
# Example 2: Unsupported engine name raises ConfigurationError
try:
    loader = DataLoader(engine="snowflake")
except ConfigurationError as exc:
    print(f"ConfigurationError: {exc}")

In [ ]:
# Example 3: Partial date filter raises ValueError (all-or-none rule)
try:
    loader = S3ParquetLoader()
    df = loader.loader(
        "leads",
        ["_id"],
        date_column="createdAt",  # missing start_date and end_date!
    )
except ValueError as exc:
    print(f"ValueError: {exc}")

In [ ]:
# Example 4: Unsupported operation gives an actionable error message
try:
    wrangler_loader = DataLoader(engine="wrangler")
    wrangler_loader.read_sql("SELECT 1")  # Wrangler can't do SQL
except WranglerError as exc:
    print(f"WranglerError: {exc}")

---
## 8. Advanced Patterns

### 8.1 Registering a custom engine

New engines can be plugged in via `LoaderFactory.register()`. Implement
the `BaseLoader` ABC and register under a name.

In [19]:
from dataSource import LoaderFactory, BaseLoader

# Check currently supported engines
print("Supported engines:", LoaderFactory.supported_engines())

Supported engines: ('athena', 'duckdb', 'wrangler')


In [20]:
# Example: registering a hypothetical custom engine
# (This is a skeleton — implement the abstract methods for a real engine)

class MyCustomLoader(BaseLoader):
    engine = "custom"
    
    def read_parquet(self, path=None, *, columns=None, **kwargs):
        raise NotImplementedError("Implement me")
    
    def read_sql(self, sql, **kwargs):
        raise NotImplementedError("Implement me")
    
    def read_table(self, table, *, columns=None, date_column=None,
                   start_date=None, end_date=None, **kwargs):
        raise NotImplementedError("Implement me")
    
    def scan(self, target, **kwargs):
        raise NotImplementedError("Implement me")

# Register it
LoaderFactory.register("custom", MyCustomLoader)
print("Supported engines after registration:", LoaderFactory.supported_engines())

# Now you can use it
# custom_loader = DataLoader(engine="custom")

Supported engines after registration: ('athena', 'custom', 'duckdb', 'wrangler')


### 8.2 Enabling debug logging

Engine modules log timing and operations under the `dataSource.*` namespace.
Enable debug logging to see what's happening behind the scenes.

In [ ]:
import logging

# Enable debug-level logging for the entire library
logging.basicConfig(level=logging.DEBUG)
logging.getLogger("dataSource").setLevel(logging.DEBUG)

# Now all engine calls will log timing information:
# loader = DataLoader()
# df = loader.read_table("leads", columns=["_id"])
# → INFO dataSource.duckdb: duckdb.read_table finished in 1.23s (4500 rows)

print("✅ Logging configured — set level to WARNING to reduce noise")

### 8.3 S3 prefix resolution helpers

Use `resolve_s3_prefix()` to understand how named sources map to S3 prefixes.

In [21]:
from dataSource import resolve_s3_prefix

# Named sources
print("leads-db →", resolve_s3_prefix("leads-db"))
print("crm      →", resolve_s3_prefix("crm"))

# Aliases work too
print("leads_db →", resolve_s3_prefix("leads_db"))    # normalized

# Explicit prefix (bypasses env lookup)
print("explicit →", resolve_s3_prefix(s3_prefix="production_raw_dump/analytics"))

leads-db → production_raw_dump/leads-db
crm      → production_raw_dump/crm
leads_db → production_raw_dump/leads-db
explicit → production_raw_dump/analytics


### 8.4 Date normalization utilities

The library exports helpers for date normalization and validation,
which are useful if you're building custom queries.

In [22]:
from dataSource import (
    normalize_date_sql,
    require_full_date_filter,
    wrap_sql_with_date_filter,
)

# Normalize various date formats to 'YYYY-MM-DD HH:MM:SS' for DuckDB
import pandas as pd
from datetime import datetime

print(normalize_date_sql("2025-06-01", "start_date"))       # string
print(normalize_date_sql(pd.Timestamp("2025-06-01"), "ts")) # Timestamp
print(normalize_date_sql(datetime(2025, 6, 1), "dt"))       # datetime

2025-06-01 00:00:00
2025-06-01 00:00:00
2025-06-01 00:00:00


In [23]:
# Validate the all-or-none date filter rule
result = require_full_date_filter("createdAt", "2025-06-01", "2025-06-07")
print("Valid filter:", result)

result = require_full_date_filter(None, None, None)
print("No filter:  ", result)

Valid filter: ('createdAt', '2025-06-01', '2025-06-07')
No filter:   None


In [24]:
# Wrap SQL with a date filter
sql = "SELECT * FROM parquet_scan('s3://bucket/data/*.parquet')"
filtered_sql = wrap_sql_with_date_filter(
    sql, "createdAt", "2025-06-01", "2025-06-07"
)
print(filtered_sql)


        SELECT * FROM (SELECT * FROM parquet_scan('s3://bucket/data/*.parquet')) AS _q
        WHERE "createdAt" >= TIMESTAMP '2025-06-01 00:00:00'
          AND "createdAt" <= TIMESTAMP '2025-06-07 00:00:00'
    


---
## 9. Tips & Gotchas

| Situation | Do this |
|-----------|---------|
| **Lead-DB vs CRM** | `S3ParquetLoader(source="leads-db")` or `source="crm"` |
| **Arbitrary dump** | `S3ParquetLoader(s3_prefix="production_raw_dump/...")` |
| **Large indexed collection** | Pass `date_column` + `start_date` + `end_date`; omit all three only for a full scan |
| **Date filter args** | All three together or none — partial args raise `ValueError` |
| **Ambiguous dates** (`01/02/2025`) | Prefer ISO `YYYY-MM-DD` |
| **Schema drift** | `loader()` skips unknown columns; watch for `[loader] columns not found` |
| **New indexed dump** | Add name to `INDEXED_COLLECTIONS` and ensure `_index/{name}_index.json` exists on S3 |
| **Large scans hitting S3 rate limits** | Switch to `DataLoader(engine="athena")` for server-side SQL |
| **Bulk dataset reads** | Use `DataLoader(engine="wrangler")` for partition filters and chunking |
| **SQL not supported (Wrangler)** | Wrangler can't run SQL; use `engine="athena"` or `engine="duckdb"` |
| **Parquet reads not supported (Athena)** | Athena queries catalog tables, not raw paths; use `engine="duckdb"` or `engine="wrangler"` |
| **Catch all errors** | `except LoaderError` catches DuckDB, Athena, and Wrangler errors |
| **Debug what's happening** | `logging.getLogger("dataSource").setLevel(logging.DEBUG)` |

---

### S3 Layout Reference

| Type | Path |
|------|------|
| Normal collection | `s3://{S3_BUCKET}/{prefix}/{collection}/*.parquet` |
| Large (indexed) | `s3://{S3_BUCKET}/{prefix}/{collection}/data/*.parquet` |
| Chunk manifest | `s3://{S3_BUCKET}/{prefix}/{collection}/_index/{collection}_index.json` |

### Credential Resolution

The library deliberately hides static env keys (`AWS_ACCESS_KEY_ID`, etc.)
while boto3 checks its default chain, so **IAM roles/profiles always win**
over `.env` keys. Static keys are only a fallback for local development.